# Stage C3.1 — ReLU Feasibility Check

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Intermediate diagnostic between **C3** and **C4**.

**What this checks:** ReLU networks are piecewise-*linear* — their
second derivative is exactly 0 almost everywhere (kinks aside, and a
generic collocation point essentially never lands exactly on one). The
HC-λ convexity penalty (Eq. 3.11) is built from that second derivative
via autodiff. So a ReLU-activated network can show a near-zero
convexity penalty **whether or not it actually learned a U-shaped
HC-λ relationship** — flat, purely monotonic, and even zigzag outputs
can all pass the same way a genuine U-shape does, because the penalty
never sees anything but 0.

**Scope, stated precisely:** this only touches the HC-λ constraint
(Eq. 3.11), the one built on a *second* derivative. The four
first-derivative constraints (Eq. 3.9, 3.10, 3.12, 3.13) use ReLU's
local slope, which is well-defined and not trivially zero — Section 7
below checks this holds, rather than assuming it.

**What "feasible" means here:** not a loss value — whether the
network's actual predicted HC(λ) curve, swept directly, decreases to a
minimum and then increases, the way Sec. 3.3.1.3's physical
justification (partial-oxidation HC at low λ, quench/incomplete
combustion at high λ) says it should.

**Input:** `data/masters_data.xlsx`, `outputs/C1_collocation_points.csv`,
`outputs/C2_loss_config.json`, `outputs/C3_trials_history.csv`
**Output:** a verdict on whether trusting C3's `best_hp` is safe if it
happens to be ReLU, plus the comparison plot.


## Setup

In [ ]:
import json
import numpy as np
import polars as pl
import plotly.graph_objects as go
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("polars    ", pl.__version__)
import plotly
print("plotly    ", plotly.__version__)
print("tensorflow", tf.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"
SEED = 42


## 1. Load data, split, collocation points, trial history

In [ ]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS
N_IN, N_OUT = len(INPUT_COLS), len(OUTPUT_COLS)
SOI_IDX, LAMBDA_IDX, SUBRATE_IDX, PRAIL_IDX = [INPUT_COLS.index(c) for c in INPUT_COLS]
HC_IDX, NOX_IDX, CO2_IDX, PM_IDX, ETA_IDX = [OUTPUT_COLS.index(c) for c in OUTPUT_COLS]

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}
deviation = np.column_stack([np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]

def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)

ofat_block = smooth_isolated_labels(raw_block)
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    pos = np.array([0.5]) if m == 1 else np.empty(m)
    if m > 1:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2
rng_np = np.random.default_rng(SEED)
jitter = rng_np.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))
split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"
df = df.with_columns(pl.Series("ofat_block", ofat_block), pl.Series("split", split))
train_df = df.filter(pl.col("split") == "train")
train_min = {c: train_df[c].min() for c in ALL_COLS}
train_max = {c: train_df[c].max() for c in ALL_COLS}
df = df.with_columns([
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
])

def split_arrays(split_name, cols):
    sub = df.filter(pl.col("split") == split_name)
    return sub.select([f"{c}_norm" for c in cols]).to_numpy().astype(np.float32)

X_train, Y_train = split_arrays("train", INPUT_COLS), split_arrays("train", OUTPUT_COLS)
X_val, Y_val = split_arrays("val", INPUT_COLS), split_arrays("val", OUTPUT_COLS)
sigma2 = np.maximum(Y_train.var(axis=0, ddof=1), 1e-8)

colloc_df = pl.read_csv(OUT_DIR / "C1_collocation_points.csv")
X_colloc = colloc_df.select(INPUT_COLS).to_numpy().astype(np.float32)
rho_colloc = colloc_df["density_ratio"].to_numpy()
validity_eff_nox = colloc_df["validity_eff_nox"].to_numpy()

with open(OUT_DIR / "C2_loss_config.json") as f:
    w_final_base = json.load(f)["w_final"]

# Optuna's study.trials_dataframe() (what C3 actually exports) prefixes
# every hyperparameter column with "params_" and calls the objective
# column "value" -- normalize both here so the rest of this notebook can
# use plain names (n_layers, activation, cv_mse, ...) instead of carrying
# that prefix through every reference below.
raw_trials = pl.read_csv(OUT_DIR / "C3_trials_history.csv")
param_cols = [c for c in raw_trials.columns if c.startswith("params_")]
rename_map = {c: c.removeprefix("params_") for c in param_cols}
if "value" in raw_trials.columns:
    rename_map["value"] = "cv_mse"
trials_df = raw_trials.rename(rename_map)
if "state" in trials_df.columns:
    before = trials_df.shape[0]
    trials_df = trials_df.filter(pl.col("state") == "COMPLETE")
    if trials_df.shape[0] < before:
        print(f"Dropped {before - trials_df.shape[0]} non-completed trial(s) (failed/pruned).")

print(f"{trials_df.shape[0]} completed trials loaded, activations present: "
      f"{trials_df['activation'].unique().to_list()}")


## 2. Select trials to inspect

Best overall, best ReLU specifically, and best non-ReLU — the last two
give a same-search, apples-to-apples comparison instead of just
eyeballing one model in isolation.

In [ ]:
sorted_trials = trials_df.sort("cv_mse")

def first_row_as_dict(frame):
    return frame.head(1).to_dicts()[0] if frame.shape[0] > 0 else None

best_overall = first_row_as_dict(sorted_trials)
best_relu = first_row_as_dict(sorted_trials.filter(pl.col("activation") == "relu"))
best_non_relu = first_row_as_dict(sorted_trials.filter(pl.col("activation") != "relu"))

for label, t in [("best overall", best_overall), ("best ReLU", best_relu), ("best non-ReLU", best_non_relu)]:
    if t is None:
        print(f"{label:15s}: none found in trial history")
    else:
        print(f"{label:15s}: activation={t['activation']:8s} layers={t['n_layers']} "
              f"units={t['n_units']}  cv_mse={t['cv_mse']:.5f}")


## 3. Retrain the selected configs — with the physics loss active

One representative model per config (not a fresh CV search), trained
on A3's original 28/6 train/val split. **Important:** this has to
replicate C3's actual training procedure — physics + regularization
once per epoch, data once per batch (the optimization from the
previous message) — not a plain data-only fit. A model trained on data
alone would answer a different, less relevant question ("does an
ordinary network happen to learn a U-shape") instead of the one this
notebook exists to answer ("does the *physics-informed* network,
trained with the same loss C3 searched over, actually learn one").

In [ ]:
CONSTRAINT_CATEGORY = {
    "NOx-SOI": "monotonic", "PM-lambda": "monotonic", "HC-lambda": "shape",
    "NOx-PM": "tradeoff", "eta-NOx": "tradeoff", "non-negativity": "nonneg",
}
CATEGORY_KEY = {"monotonic": "scale_monotonic", "shape": "scale_shape",
                 "tradeoff": "scale_tradeoff", "nonneg": "scale_nonneg"}

def build_from_trial(t, seed):
    tf.random.set_seed(seed)
    model = keras.Sequential([keras.Input(shape=(N_IN,))])
    for _ in range(int(t["n_layers"])):
        model.add(layers.Dense(int(t["n_units"]), activation=t["activation"]))
        if t["dropout_rate"] > 0:
            model.add(layers.Dropout(float(t["dropout_rate"])))
    model.add(layers.Dense(N_OUT, activation="linear"))
    return model

def monotonic_decreasing_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x_t)
        target = predict_fn(x_t)[:, out_idx]
    d = tape.gradient(target, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, d))

def convexity_per_point(predict_fn, x, out_idx, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape() as tape2:
        tape2.watch(x_t)
        with tf.GradientTape() as tape1:
            tape1.watch(x_t)
            target = predict_fn(x_t)[:, out_idx]
        d_first = tape1.gradient(target, x_t)[:, in_idx]
    d_second = tape2.gradient(d_first, x_t)[:, in_idx]
    return tf.square(tf.maximum(0.0, -d_second))

def tradeoff_per_point(predict_fn, x, out_idx_a, out_idx_b, in_idx):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    with tf.GradientTape(persistent=True) as tape:
        tape.watch(x_t)
        y = predict_fn(x_t)
        a, b = y[:, out_idx_a], y[:, out_idx_b]
    grad_a = tape.gradient(a, x_t)[:, in_idx]
    grad_b = tape.gradient(b, x_t)[:, in_idx]
    del tape
    return tf.square(tf.maximum(0.0, grad_a * grad_b))

def nonneg_loss(predict_fn, x, out_idxs=(HC_IDX, NOX_IDX, CO2_IDX, PM_IDX)):
    x_t = tf.convert_to_tensor(x, dtype=tf.float32)
    y = predict_fn(x_t)
    emissions = tf.gather(y, list(out_idxs), axis=1)
    return tf.reduce_mean(tf.reduce_sum(tf.square(tf.maximum(0.0, -emissions)), axis=1))

def weighted_physics_loss(per_point, lambda_x):
    return tf.reduce_mean(tf.constant(lambda_x, dtype=tf.float32) * per_point)

def data_loss(predict_fn, X, Y):
    pred = predict_fn(tf.convert_to_tensor(X, dtype=tf.float32))
    sq_err = tf.square(pred - tf.constant(Y)) / tf.constant(sigma2, dtype=tf.float32)
    return tf.reduce_mean(tf.reduce_sum(sq_err, axis=1))

def regularization_loss(model):
    P = model.count_params()
    sq_sum = tf.add_n([tf.reduce_sum(tf.square(w)) for w in model.trainable_weights if len(w.shape) > 1])
    return sq_sum / P

def physics_and_reg_step(model, optimizer, w_scaled, weight_decay):
    predict_fn = lambda x: model(x, training=True)
    with tf.GradientTape() as tape:
        per_point = {
            "NOx-SOI": monotonic_decreasing_per_point(predict_fn, X_colloc, NOX_IDX, SOI_IDX),
            "PM-lambda": monotonic_decreasing_per_point(predict_fn, X_colloc, PM_IDX, LAMBDA_IDX),
            "HC-lambda": convexity_per_point(predict_fn, X_colloc, HC_IDX, LAMBDA_IDX),
            "NOx-PM": tradeoff_per_point(predict_fn, X_colloc, NOX_IDX, PM_IDX, SOI_IDX),
            "eta-NOx": tradeoff_per_point(predict_fn, X_colloc, ETA_IDX, NOX_IDX, SOI_IDX),
        }
        l_phys = 0.0
        for name, pp in per_point.items():
            validity = validity_eff_nox if name == "eta-NOx" else np.ones(len(X_colloc))
            lambda_x = rho_colloc * validity
            l_phys = l_phys + float(w_scaled[name]) * weighted_physics_loss(pp, lambda_x)
        l_phys = l_phys + float(w_scaled["non-negativity"]) * nonneg_loss(predict_fn, X_colloc)
        loss = l_phys + weight_decay * regularization_loss(model)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

def data_step(model, optimizer, X, Y):
    with tf.GradientTape() as tape:
        pred = model(X, training=True)
        loss = tf.reduce_mean(tf.reduce_sum(
            tf.square(pred - tf.constant(Y)) / tf.constant(sigma2, dtype=tf.float32), axis=1))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

def train_one(t, seed, max_epochs=400, patience=40):
    model = build_from_trial(t, seed)
    optimizer = keras.optimizers.Adam(learning_rate=float(t["learning_rate"]),
                                       beta_1=float(t["beta_1"]), beta_2=float(t["beta_2"]))
    w_scaled = {name: w_final_base[name] * t[CATEGORY_KEY[CONSTRAINT_CATEGORY[name]]]
                for name in w_final_base}
    bs = int(t["batch_size"])

    best_val, no_improve, best_weights = np.inf, 0, None
    for epoch in range(max_epochs):
        physics_and_reg_step(model, optimizer, w_scaled, float(t["weight_decay"]))
        idx = np.random.default_rng(seed + epoch).permutation(len(X_train))
        for start in range(0, len(idx), bs):
            batch = idx[start:start + bs]
            data_step(model, optimizer, X_train[batch], Y_train[batch])
        val_loss = float(data_loss(lambda x: model(x, training=False), X_val, Y_val))
        if val_loss < best_val - 1e-6:
            best_val, no_improve, best_weights = val_loss, 0, [w.numpy().copy() for w in model.trainable_weights]
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    if best_weights is not None:
        for w, bw in zip(model.trainable_weights, best_weights):
            w.assign(bw)
    return model

models = {}
for label, t in [("best_overall", best_overall), ("best_relu", best_relu), ("best_non_relu", best_non_relu)]:
    if t is None:
        continue
    print(f"training {label} ({t['activation']})...")
    keras.backend.clear_session()
    models[label] = {"model": train_one(t, seed=SEED), "trial": t}
print("done:", list(models.keys()))


## 4. The loophole, confirmed numerically

Reproduces Eq. 3.11's penalty exactly as C1/C2/C3 compute it — on the
1000 real collocation points, using each model's own weights.

In [ ]:
rows = []
for label, m in models.items():
    predict_fn = lambda x, mdl=m["model"]: mdl(x, training=False)
    penalty = float(tf.reduce_mean(convexity_per_point(predict_fn, X_colloc, HC_IDX, LAMBDA_IDX)))
    rows.append({"model": label, "activation": m["trial"]["activation"], "hc_lambda_penalty": penalty})
penalty_table = pl.DataFrame(rows)
penalty_table


## 5. The real test — sweep HC vs. λ directly

Vary λ across its full normalized range, holding SOI/sub_rate/P_rail
at their training-set median (a representative baseline point, same
idea as A3's OFAT baseline) — this is activation-agnostic: it looks at
what the network actually outputs, not at a derivative that ReLU can
trivially zero out.

Two checks, robust to realistic prediction noise (not exact-zero
tolerances — see chat for why an earlier stricter version of this
failed on a genuinely U-shaped curve):
- **`looks_u_shaped`**: does the curve decrease to a genuine *interior*
  minimum (not sitting at either edge, which would just mean
  monotonic) and then increase, allowing an 8% fraction of noisy steps?
- **finite-difference convexity**: a discrete second difference over
  the swept grid — unlike autodiff at a single point, a finite step
  size actually crosses ReLU's kinks, so this sees curvature autodiff
  structurally cannot.

In [ ]:
def looks_u_shaped(y, edge_margin=0.05, violation_tol=0.08):
    i_min = int(np.argmin(y))
    n_pts = len(y)
    if i_min < edge_margin * n_pts or i_min > (1 - edge_margin) * n_pts:
        return False, i_min
    before, after = y[:i_min + 1], y[i_min:]
    frac_bad_before = np.mean(np.diff(before) > 0) if len(before) > 1 else 0.0
    frac_bad_after = np.mean(np.diff(after) < 0) if len(after) > 1 else 0.0
    return bool(frac_bad_before <= violation_tol and frac_bad_after <= violation_tol), i_min

def finite_diff_violation_rate(y, tol=1e-6):
    d2 = y[2:] - 2 * y[1:-1] + y[:-2]
    return float(np.mean(d2 < -tol))

N_SWEEP = 300
lam_sweep = np.linspace(0, 1, N_SWEEP).astype(np.float32)
baseline = np.array([df[f"{c}_norm"].median() for c in INPUT_COLS], dtype=np.float32)

sweep_results = {}
for label, m in models.items():
    X_sweep = np.tile(baseline, (N_SWEEP, 1))
    X_sweep[:, LAMBDA_IDX] = lam_sweep
    hc_sweep = m["model"](X_sweep, training=False).numpy()[:, HC_IDX]
    u_shaped, i_min = looks_u_shaped(hc_sweep)
    viol_rate = finite_diff_violation_rate(hc_sweep)
    sweep_results[label] = {"hc": hc_sweep, "u_shaped": u_shaped, "min_at": lam_sweep[i_min],
                             "violation_rate": viol_rate, "activation": m["trial"]["activation"]}
    print(f"{label:15s} ({m['trial']['activation']:8s}): looks_u_shaped={u_shaped!s:5s}  "
          f"min at lambda_norm={lam_sweep[i_min]:.2f}  finite-diff violation rate={viol_rate:.3f}")


## 6. Visual comparison — swept curves plus the real λ-sweep experimental points

The real overlay comes from the actual λ-sweep block A1/A3 already
identify (`ofat_block == "lambda"`) — the rows where λ was the swept
variable, so this is a like-for-like reference, not an arbitrary
scatter of unrelated points.

In [ ]:
lam_block = df.filter(pl.col("ofat_block") == "lambda")
real_lam = lam_block["lambda_norm"].to_numpy()
real_hc = lam_block["HC_norm"].to_numpy()

colors = {"best_overall": "#185FA5", "best_relu": "#993C1D", "best_non_relu": "#3B6D11"}
fig = go.Figure()
fig.add_trace(go.Scatter(x=real_lam, y=real_hc, mode="markers",
                          marker=dict(color="#5A5A55", size=9, symbol="x"),
                          name="real data (lambda-swept rows)"))
for label, res in sweep_results.items():
    fig.add_trace(go.Scatter(x=lam_sweep, y=res["hc"], mode="lines",
                              line=dict(color=colors.get(label, "#000"), width=2.5),
                              name=f"{label} ({res['activation']})"))
fig.update_layout(title="HC vs. lambda (others held at median) -- swept prediction vs. real data",
                   xaxis_title="lambda (normalized)", yaxis_title="HC (normalized)",
                   width=850, height=480)
fig.show()


## 7. Scope check — do the first-derivative constraints hold up under ReLU?

Confirms this is specifically a second-derivative problem, not a
blanket "ReLU breaks physics constraints" claim.

In [ ]:
if models.get("best_relu") is not None:
    predict_fn = lambda x: models["best_relu"]["model"](x, training=False)
    nox_soi_penalty = float(tf.reduce_mean(monotonic_decreasing_per_point(predict_fn, X_colloc, NOX_IDX, SOI_IDX)))
    pm_lambda_penalty = float(tf.reduce_mean(monotonic_decreasing_per_point(predict_fn, X_colloc, PM_IDX, LAMBDA_IDX)))
    print(f"best_relu model, first-derivative constraints (non-trivial for ReLU, unlike HC-lambda):")
    print(f"  NOx-SOI monotonic penalty: {nox_soi_penalty:.5f}  (varies with actual learned slope, not structurally 0)")
    print(f"  PM-lambda monotonic penalty: {pm_lambda_penalty:.5f}")
else:
    print("No ReLU trial found in this search's history -- nothing to check here.")


## 8. Verdict

In [ ]:
print("=" * 70)
for label, res in sweep_results.items():
    verdict = "genuinely U-shaped" if res["u_shaped"] else "NOT U-shaped despite the constraint"
    flag = "" if res["u_shaped"] else "  <-- do not trust this one's HC-lambda physics compliance"
    print(f"{label:15s} ({res['activation']:8s}): {verdict}{flag}")
print("=" * 70)

relu_ok = sweep_results.get("best_relu", {}).get("u_shaped")
if relu_ok is False:
    print("\nRecommendation: if C3's best_hp (Section 8 of C3) has activation='relu',")
    print("do not carry it into C4 without checking this notebook's Section 6 plot first.")
    print("Options: exclude 'relu' from the search space and rerun C3, or keep it but")
    print("add this sweep-based check as a post-hoc filter on which trials are eligible")
    print("to be selected as the winner.")
elif relu_ok is True:
    print("\nThis particular best-ReLU trial happens to have learned a genuine U-shape --")
    print("the loophole exists in principle (Section 4) but didn't bite this specific run.")
    print("Re-check if C3 is rerun with a different seed or search budget.")


## Optional — persist outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sweep_export = pl.DataFrame({
    "lambda_norm": lam_sweep,
    **{f"HC_{label}": res["hc"] for label, res in sweep_results.items()},
})
sweep_export.write_csv(OUT_DIR / "C3.1_hc_lambda_sweep.csv")

verdict_export = pl.DataFrame([
    {"model": label, "activation": res["activation"], "looks_u_shaped": res["u_shaped"],
     "min_at_lambda_norm": res["min_at"], "finite_diff_violation_rate": res["violation_rate"]}
    for label, res in sweep_results.items()
])
verdict_export.write_csv(OUT_DIR / "C3.1_verdict.csv")
print(f"Saved to {OUT_DIR}")


## Next

If the verdict above flags the best ReLU trial as not genuinely
U-shaped, the cleanest fix is upstream, in **C3**: either drop `"relu"`
from `SEARCH_SPACE["activation"]` and rerun the search (simplest), or
keep it and add this notebook's `looks_u_shaped` check as a filter
Optuna's objective applies before a trial can win, not just as a
post-hoc report. Either way, **C4** should build from whichever
`best_hp` this notebook gives a clean bill of health to, not
necessarily the raw C3 output.
